In [ ]:
def main(datasources, start_date, end_date):
    """AI V10: multi-island clientele reversal with top-purity reshaping."""
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    query_start = (start_ts - pd.Timedelta(days=90)).strftime("%Y-%m-%d %H:%M:%S")

    def weighted_delta(left, field, right):
        return " + ".join(
            f"{w}.0 * ({left}_{field}{i} - {right}_{field}{i})"
            for i, w in zip(range(1, 6), range(5, 0, -1))
        )

    def weighted_total(field):
        return " + ".join(
            f"{w}.0 * (bid_{field}{i} + ask_{field}{i})"
            for i, w in zip(range(1, 6), range(5, 0, -1))
        )

    book_num = weighted_delta("bid", "volume", "ask")
    book_den = weighted_total("volume")
    order_num = weighted_delta("bid", "num_orders", "ask")
    order_den = weighted_total("num_orders")

    minute_sql = f"""
    WITH m AS (
        SELECT
            date,
            date::DATE::DATETIME AS d,
            instrument,
            open,
            high,
            low,
            close,
            volume,
            amount,
            CASE WHEN ({book_den}) > 0 THEN ({book_num}) / ({book_den}) ELSE NULL END AS book_imb,
            CASE WHEN ({order_den}) > 0 THEN ({order_num}) / ({order_den}) ELSE NULL END AS order_imb,
            CASE
                WHEN ask_price1 > 0 AND bid_price1 > 0
                 AND ask_price1 >= bid_price1
                 AND (ask_price1 + bid_price1) > 0
                THEN (ask_price1 - bid_price1) / ((ask_price1 + bid_price1) / 2.0)
                ELSE NULL
            END AS spread
        FROM {bar1m}
        WHERE open > 0
          AND high > 0
          AND low > 0
          AND close > 0
          AND volume >= 0
          AND amount >= 0
    ),
    daily AS (
        SELECT
            d AS date,
            instrument,
            FIRST(open ORDER BY date) AS open_price,
            LAST(close ORDER BY date) AS close_price,
            MAX(high) AS high_price,
            MIN(low) AS low_price,
            SUM(amount) AS amount_sum,
            SUM(volume) AS volume_sum,
            CASE WHEN SUM(volume) > 0 THEN SUM(amount) / SUM(volume) ELSE NULL END AS vwap,
            LAST(close ORDER BY date) FILTER (WHERE strftime(date, '%H:%M:%S') < '10:00:00') AS close_30,
            MAX(high) FILTER (WHERE strftime(date, '%H:%M:%S') < '10:00:00') AS high_30,
            MIN(low) FILTER (WHERE strftime(date, '%H:%M:%S') < '10:00:00') AS low_30,
            SUM(amount) FILTER (WHERE strftime(date, '%H:%M:%S') < '10:00:00') AS amount_30,
            LAST(close ORDER BY date) FILTER (WHERE strftime(date, '%H:%M:%S') < '11:00:00') AS close_1100,
            SUM(amount) FILTER (WHERE strftime(date, '%H:%M:%S') >= '10:00:00' AND strftime(date, '%H:%M:%S') < '11:00:00') AS amount_mid_morning,
            AVG(book_imb) AS book_all,
            AVG(order_imb) AS order_all,
            AVG(spread) AS spread_all,
            AVG(book_imb) FILTER (WHERE strftime(date, '%H:%M:%S') < '10:00:00') AS book_30,
            AVG(order_imb) FILTER (WHERE strftime(date, '%H:%M:%S') < '10:00:00') AS order_30,
            AVG(spread) FILTER (WHERE strftime(date, '%H:%M:%S') < '10:00:00') AS spread_30,
            AVG(book_imb) FILTER (WHERE strftime(date, '%H:%M:%S') >= '13:00:00' AND strftime(date, '%H:%M:%S') < '14:00:00') AS book_pm1,
            AVG(order_imb) FILTER (WHERE strftime(date, '%H:%M:%S') >= '13:00:00' AND strftime(date, '%H:%M:%S') < '14:00:00') AS order_pm1,
            AVG(book_imb) FILTER (WHERE strftime(date, '%H:%M:%S') >= '14:30:00') AS book_tail,
            AVG(order_imb) FILTER (WHERE strftime(date, '%H:%M:%S') >= '14:30:00') AS order_tail,
            AVG(spread) FILTER (WHERE strftime(date, '%H:%M:%S') >= '14:30:00') AS spread_tail,
            SUM(amount) FILTER (WHERE strftime(date, '%H:%M:%S') >= '14:30:00') AS amount_tail,
            LAST(close ORDER BY date) FILTER (WHERE strftime(date, '%H:%M:%S') >= '14:30:00') AS close_tail
        FROM m
        GROUP BY d, instrument
    )
    SELECT * FROM daily ORDER BY date, instrument
    """

    df = dai.query(minute_sql, filters={"date": [query_start, end_date]}, compression=True).df()
    if df.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    df["date"] = pd.to_datetime(df["date"])
    for col in [c for c in df.columns if c not in ("date", "instrument")]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.sort_values(["instrument", "date"]).copy()

    df["prev_close"] = df.groupby("instrument")["close_price"].shift(1)
    df["open_gap"] = df["open_price"] / df["prev_close"] - 1.0
    df["intraday_ret"] = df["close_price"] / df["open_price"] - 1.0
    df["first30_ret"] = df["close_30"] / df["open_price"] - 1.0
    df["mid_repair"] = df["close_1100"] / df["close_30"] - 1.0
    df["tail_ret"] = df["close_price"] / df["close_tail"] - 1.0
    df["vwap_gap"] = df["vwap"] / df["close_price"] - 1.0
    df["range_pct"] = (df["high_price"] - df["low_price"]) / df["close_price"]
    df["range_30"] = (df["high_30"] - df["low_30"]) / df["open_price"]
    df["close_location"] = (df["close_price"] - df["low_price"]) / (df["high_price"] - df["low_price"]).replace(0, np.nan)
    df["amount_30_share"] = df["amount_30"] / df["amount_sum"].replace(0, np.nan)
    df["amount_tail_share"] = df["amount_tail"] / df["amount_sum"].replace(0, np.nan)
    df["tail_book_delta"] = df["book_tail"] - df["book_all"]
    df["tail_order_delta"] = df["order_tail"] - df["order_all"]
    df["tail_spread_repair"] = df["spread_all"] - df["spread_tail"]
    df["pm_to_tail_book"] = df["book_tail"] - df["book_pm1"]

    def cs_rank(col):
        return df.groupby("date")[col].transform(lambda s: s.rank(method="average", pct=True) - 0.5).fillna(0.0)

    open_extreme = cs_rank("open_gap").abs()
    first30_extreme = cs_rank("first30_ret").abs()
    early_crowd = cs_rank("amount_30_share") + 0.5 * cs_rank("spread_30")

    df["open_shock_reversal"] = (
        -0.55 * cs_rank("open_gap")
        + 0.30 * cs_rank("intraday_ret")
        - 0.15 * open_extreme * cs_rank("amount_30_share")
    )
    df["first30_mid_reversal"] = (
        -0.50 * cs_rank("first30_ret")
        + 0.38 * cs_rank("mid_repair")
        - 0.18 * first30_extreme
        - 0.10 * early_crowd
    )
    df["liquidity_fragility"] = -(
        0.38 * cs_rank("spread_all")
        + 0.30 * cs_rank("range_pct")
        + 0.20 * cs_rank("vwap_gap").abs()
        + 0.12 * cs_rank("amount_30_share").abs()
    )
    df["lottery_overheat_penalty"] = -(
        0.40 * cs_rank("range_pct")
        + 0.34 * cs_rank("close_location")
        + 0.16 * cs_rank("first30_ret")
        + 0.10 * cs_rank("amount_30_share")
    )
    df["institutional_close_confirm"] = (
        0.34 * cs_rank("tail_book_delta")
        + 0.22 * cs_rank("tail_order_delta")
        + 0.20 * cs_rank("tail_spread_repair")
        + 0.14 * cs_rank("pm_to_tail_book")
        - 0.10 * cs_rank("tail_ret").abs()
    )

    df["clientele_reversal_core"] = (
        0.26 * df["open_shock_reversal"]
        + 0.25 * df["first30_mid_reversal"]
        + 0.20 * df["liquidity_fragility"]
        + 0.18 * df["lottery_overheat_penalty"]
        + 0.11 * df["institutional_close_confirm"]
    )
    df["clientele_reversal_smooth"] = df.groupby("instrument")["clientele_reversal_core"].transform(
        lambda s: s.rolling(2, min_periods=1).mean()
    )
    df["linear_clientele_score"] = 0.86 * df["clientele_reversal_core"] + 0.14 * df["clientele_reversal_smooth"]

    linear_rank = df.groupby("date")["linear_clientele_score"].transform(
        lambda s: s.rank(method="average", pct=True) - 0.5
    ).fillna(0.0)
    heat_rank = df.groupby("date")["lottery_overheat_penalty"].transform(
        lambda s: s.rank(method="average", pct=True) - 0.5
    ).fillna(0.0)
    fragility_rank = df.groupby("date")["liquidity_fragility"].transform(
        lambda s: s.rank(method="average", pct=True) - 0.5
    ).fillna(0.0)
    close_confirm_rank = df.groupby("date")["institutional_close_confirm"].transform(
        lambda s: s.rank(method="average", pct=True) - 0.5
    ).fillna(0.0)

    hump_anchor = -0.18
    df["moderate_reversal_score"] = -((linear_rank - hump_anchor).abs())
    df["tail_shape_guard"] = -0.65 * (linear_rank.clip(lower=0.12) - 0.12).pow(2)
    df["balanced_clientele_score"] = (
        0.58 * df["moderate_reversal_score"]
        + 0.18 * heat_rank
        + 0.14 * fragility_rank
        + 0.10 * close_confirm_rank
        + df["tail_shape_guard"]
    )
    df["balanced_clientele_smooth"] = df.groupby("instrument")["balanced_clientele_score"].transform(
        lambda s: s.rolling(2, min_periods=1).mean()
    )
    df["v8_base_score"] = 0.88 * df["balanced_clientele_score"] + 0.12 * df["balanced_clientele_smooth"]

    base_rank = df.groupby("date")["v8_base_score"].transform(
        lambda s: s.rank(method="average", pct=True)
    ).fillna(0.5)
    rank_centered = base_rank - 0.5
    df["base_rank"] = rank_centered
    df["quality_corridor_score"] = -((base_rank - 0.68).abs())
    df["top_tail_boost"] = ((base_rank - 0.92).clip(lower=0.0) / 0.08).pow(2)
    df["shoulder_penalty"] = np.exp(-((base_rank - 0.84) / 0.055).pow(2))
    df["left_tail_penalty"] = ((0.12 - base_rank).clip(lower=0.0) / 0.12).pow(2)
    df["shoulder_suppressed_score"] = (
        0.36 * rank_centered
        + 0.30 * df["quality_corridor_score"]
        + 0.22 * df["top_tail_boost"]
        + 0.14 * heat_rank
        + 0.10 * fragility_rank
        + 0.08 * close_confirm_rank
        + 0.05 * linear_rank
        - 0.34 * df["shoulder_penalty"]
        - 0.16 * df["left_tail_penalty"]
    )

    v9_rank = df.groupby("date")["shoulder_suppressed_score"].transform(
        lambda s: s.rank(method="average", pct=True)
    ).fillna(0.5)
    df["v9_rank"] = v9_rank - 0.5
    df["island_mid_score"] = np.exp(-((v9_rank - 0.50) / 0.085).pow(2))
    df["island_upper_score"] = np.exp(-((v9_rank - 0.78) / 0.070).pow(2))
    df["island_tail_score"] = np.exp(-((v9_rank - 0.965) / 0.050).pow(2))
    df["bad_low_penalty"] = (
        0.70 * np.exp(-((v9_rank - 0.10) / 0.110).pow(2))
        + 0.30 * np.exp(-((v9_rank - 0.22) / 0.085).pow(2))
    )
    df["bad_middle_penalty"] = np.exp(-((v9_rank - 0.58) / 0.060).pow(2))
    df["bad_upper_penalty"] = np.exp(-((v9_rank - 0.89) / 0.045).pow(2))
    stability_anchor = df.groupby("instrument")["shoulder_suppressed_score"].transform(
        lambda s: s.rolling(3, min_periods=1).mean()
    )
    stability_rank = df.groupby("date").apply(
        lambda g: stability_anchor.loc[g.index].rank(method="average", pct=True) - 0.5
    ).reset_index(level=0, drop=True).reindex(df.index).fillna(0.0)
    df["multi_island_score"] = (
        0.25 * df["island_mid_score"]
        + 0.27 * df["island_upper_score"]
        + 0.25 * df["island_tail_score"]
        + 0.18 * df["v9_rank"]
        + 0.12 * heat_rank
        + 0.08 * fragility_rank
        + 0.06 * close_confirm_rank
        + 0.05 * linear_rank
        - 0.36 * df["bad_low_penalty"]
        - 0.20 * df["bad_middle_penalty"]
        - 0.30 * df["bad_upper_penalty"]
    )
    df["anti_overfit_blend"] = (
        0.62 * df["multi_island_score"]
        + 0.24 * df["shoulder_suppressed_score"]
        + 0.14 * stability_rank
    )
    df["multi_island_smooth"] = df.groupby("instrument")["anti_overfit_blend"].transform(
        lambda s: s.rolling(2, min_periods=1).mean()
    )
    df["raw"] = 0.88 * df["anti_overfit_blend"] + 0.12 * df["multi_island_smooth"]

    exposure = dai.query(
        """
        SELECT
            date,
            instrument,
            industry_level1_code,
            BETA,
            SIZE,
            MOMENTUM,
            RESVOL,
            BTOP,
            LIQUIDTY,
            EARNYILD,
            GROWTH,
            LEVERAGE
        FROM bigalpha_2026_exposure
        ORDER BY date, instrument
        """,
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    exposure["date"] = pd.to_datetime(exposure["date"])
    df = pd.merge(df, exposure, how="left", on=["date", "instrument"])
    style_cols = ["BETA", "SIZE", "MOMENTUM", "RESVOL", "BTOP", "LIQUIDTY", "EARNYILD", "GROWTH", "LEVERAGE"]
    for col in style_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["raw"] = df.groupby(["date", "industry_level1_code"])["raw"].transform(lambda s: s - s.mean()).fillna(df["raw"])

    def residualize(g):
        y = pd.to_numeric(g["raw"], errors="coerce").to_numpy(float)
        x = g[style_cols].to_numpy(float)
        finite_y = np.isfinite(y)
        valid = finite_y & np.isfinite(x).all(axis=1)
        ans = np.full(len(g), np.nan)
        if valid.sum() >= len(style_cols) + 40:
            xx = np.column_stack([np.ones(valid.sum()), x[valid]])
            beta = np.linalg.lstsq(xx, y[valid], rcond=None)[0]
            ans[valid] = y[valid] - xx.dot(beta)
            ans[finite_y & ~valid] = y[finite_y & ~valid]
        else:
            ans[finite_y] = y[finite_y]
        return pd.Series(ans, index=g.index)

    df["factor_raw"] = df.groupby("date", group_keys=False).apply(residualize)
    raw = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)][["date", "instrument", "factor_raw"]].copy()
    raw = raw.rename(columns={"factor_raw": "factor"})

    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments ORDER BY date, instrument",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    pool["date"] = pd.to_datetime(pool["date"])
    out = pd.merge(pool, raw, how="left", on=["date", "instrument"])
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    out["factor"] = out.groupby("date")["factor"].transform(lambda s: s.fillna(s.median())).fillna(0.0)
    return out[["date", "instrument", "factor"]]

main.factor_version = "AI_V10_MULTI_ISLAND_20260803"